# 02 - Función de Pérdida Exponencial, Margen y el Algoritmo Multiclase SAMME

## Curso de Machine Learning: Modelos de Clasificación

En el año 2000, Jerome Friedman, Trevor Hastie y Robert Tibshirani demostraron un resultado histórico: **AdaBoost es equivalente a un modelo aditivo por etapas (*Forward Stagewise Additive Modeling*) que minimiza la Función de Pérdida Exponencial**. Además, para extender AdaBoost a problemas con $K > 2$ clases, Zhu et al. (2009) desarrollaron el algoritmo **SAMME** (*Stagewise Additive Modeling using a Multi-class Exponential loss function*), utilizado en Scikit-Learn.

---

### Contenidos
1. Interpretación Estadística: AdaBoost como Minimización de la Pérdida Exponencial
2. Comparativa Geométrica de Funciones de Pérdida: 0-1 Loss, Exponencial, Log-Loss y Hinge Loss
3. Extensión Multiclase: El Algoritmo **SAMME** y el término $\ln(K - 1)$
4. Evolución del Error por Etapa (`staged_score`) y Calibración de Probabilidades
5. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. La Función de Pérdida Exponencial frente a Otras Pérdidas de Clasificación

Definimos el **margen funcional** de una observación como $m_i = y_i F(\mathbf{x}_i)$, donde $y_i \in \{-1, +1\}$:
- Si $y_i F(\mathbf{x}_i) > 0$, la predicción es **correcta**.
- Si $y_i F(\mathbf{x}_i) < 0$, la predicción es **incorrecta**.

AdaBoost minimiza la **Pérdida Exponencial**:
$$\mathcal{L}_{\text{exp}}(y, F(\mathbf{x})) = \exp\left( -y \cdot F(\mathbf{x}) \right)$$

Observa en la gráfica siguiente cómo la pérdida exponencial crece mucho más agresivamente que la pérdida logística (*Log-Loss* de Gradient Boosting) o *Hinge Loss* (SVM) para márgenes negativos ($y F(\mathbf{x}) < 0$). Esto explica por qué AdaBoost es tan eficaz corrigiendo errores pero a la vez **sensible a outliers o etiquetas con ruido**.


In [ ]:
# Comparativa grafica de funciones de perdida en funcion del margen y * F(x)
margen = np.linspace(-2.2, 2.2, 400)

loss_01 = (margen < 0).astype(float)
loss_exp = np.exp(-margen)
loss_logistic = np.log2(1 + np.exp(-margen)) # Escalada en base 2 para pasar por (0, 1)
loss_hinge = np.maximum(0, 1 - margen)

plt.figure(figsize=(9, 5))
plt.plot(margen, loss_01, 'k--', lw=1.8, label='0-1 Loss (Clasificacion Ideal)')
plt.plot(margen, loss_exp, color='#d62728', lw=2.5, label='Exponential Loss (AdaBoost)')
plt.plot(margen, loss_logistic, color='#1f77b4', lw=2.2, label='Log-Loss / Binomial Deviance (Gradient Boosting)')
plt.plot(margen, loss_hinge, color='#2ca02c', lw=2.2, label='Hinge Loss (SVM)')
plt.ylim(-0.1, 4.5)
plt.xlabel('Margen Funcional: y * F(x)  [< 0: Error | > 0: Acierto]', fontweight='bold')
plt.ylabel('Penalizacion de la Funcion de Perdida L(y, F(x))', fontweight='bold')
plt.title('Comparacion de Funciones de Perdida en Algoritmos de Clasificacion', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Clasificación Multiclase: El Algoritmo SAMME

En clasificación binaria ($K=2$), un clasificador débil necesita un error $\varepsilon_m < 1/2$ para tener un peso positivo $\alpha_m > 0$. Sin embargo, cuando hay $K > 2$ clases, el azar puro acierta con probabilidad $1/K$ (es decir, un error menor a $\frac{K-1}{K}$ ya supera al azar).

El algoritmo **SAMME** (por defecto en las versiones recientes de Scikit-Learn mediante `algorithm='SAMME'`) corrige la fórmula del peso $\alpha_m$ añadiendo el término $\ln(K - 1)$:

$$\alpha_m = \eta \cdot \left[ \ln\left( \frac{1 - \varepsilon_m}{\varepsilon_m} \right) + \ln(K - 1) \right]$$

De este modo, basta con que la exactitud del árbol débil supere $1/K$ para que $\alpha_m > 0$ y contribuya positivamente al ensamble.


In [ ]:
# Demostracion de SAMME en un problema de 10 clases (Dataset Digits, K=10)
digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target)

# Con K=10 clases, un arbol de profundidad 2 como base learner captura interacciones necesarias
ada_multi = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
    n_estimators=120,
    learning_rate=0.5,
    random_state=42
)
ada_multi.fit(X_tr, y_tr)

acc_train_staged = list(ada_multi.staged_score(X_tr, y_tr))
acc_test_staged = list(ada_multi.staged_score(X_te, y_te))

print(f"Numero de clases K = {len(digits.target_names)} (Azar puro = 10.0%)")
print(f"Exactitud del 1er Arbol Debil en Test: {acc_test_staged[0]*100:.2f}%")
print(f"Exactitud Final del Ensamble (120 arboles) en Test: {acc_test_staged[-1]*100:.2f}%")

plt.figure(figsize=(9, 4.5))
plt.plot(range(1, 121), acc_train_staged, label='Exactitud Train (staged_score)', color='#1f77b4', lw=2)
plt.plot(range(1, 121), acc_test_staged, label='Exactitud Test (staged_score)', color='#2ca02c', lw=2)
plt.xlabel('Numero de Estimadores (n_estimators)', fontweight='bold')
plt.ylabel('Exactitud Multiclase (K=10)', fontweight='bold')
plt.title('Convergencia del Algoritmo SAMME en Clasificacion de 10 Clases (Digits)', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()
